# Ejercicio 06 · VAE, evaluación y Model Card

Consolida la evidencia de LAB10 en un producto ejecutable: autoencoder (AE) y autoencoder variacional
(VAE) sobre Fashion-MNIST, con reconstrucción, muestreo, interpolación, coste y límites.

**Pregunta.** ¿Qué se gana y qué se pierde al pasar de un autoencoder determinista a un VAE con
espacio latente de 2 dimensiones, y qué puede y qué no puede afirmarse sobre las muestras
generadas?

**Datos y licencia.** Fashion-MNIST (Zalando, MIT), 60.000 imágenes de entrenamiento y 10.000 de
prueba en escala de grises de 28×28. No contiene datos personales; **no se redistribuye**: se
descarga con `tf.keras.datasets` y no se versiona.

**Alcance.** Es un benchmark didáctico: los resultados describen el comportamiento del modelo en ese
dataset, no en imágenes médicas, de vigilancia ni de documentos.

In [1]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from inf8239_u03_gen.data import normalize_images, validate_images
from inf8239_u03_gen.models import build_autoencoder, build_encoder_decoder

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
REPORTS = ROOT / "reports"
MODELS = ROOT / "models"
REPORTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)
print("Python", sys.version.split()[0], "| TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

I0000 00:00:1793620571.614900   12909 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793620571.615285   12909 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793620571.650215   12909 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1793620572.463355   12909 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793620572.463579   12909 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Python 3.12.15 | TensorFlow 2.21.0 | GPU: []


E0000 00:00:1793620573.175059   12909 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 1. Datos y preprocesamiento

Las imágenes se normalizan a [0,1] y la partición de prueba (10.000) se reserva para la evaluación
final. El núcleo del panel es una **muestra aleatoria con semilla fija**: no se eligen los mejores
casos a mano.

In [2]:
(x_train_raw, y_train), (x_test_raw, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = normalize_images(x_train_raw)
x_test = normalize_images(x_test_raw)
validate_images(x_train[:100])
validate_images(x_test[:100])
print("Entrenamiento:", x_train.shape, "| Prueba:", x_test.shape)
print("Clases:", np.unique(y_train, return_counts=True))

rng = np.random.default_rng(SEED)
indice = rng.choice(len(x_test), 1000, replace=False)
x_eval = x_test[indice]
panel_index = np.arange(16)          # posiciones dentro del subconjunto de evaluación
print("Submuestra de evaluación:", x_eval.shape, "| panel:", len(panel_index))
print("Ejemplos de prueba elegidos para el panel:", indice[:16].tolist())

Entrenamiento: (60000, 28, 28, 1) | Prueba: (10000, 28, 28, 1)
Clases: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=uint8), array([6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000]))
Submuestra de evaluación: (1000, 28, 28, 1) | panel: 16
Ejemplos de prueba elegidos para el panel: [7006, 7326, 4517, 6283, 3655, 2619, 8318, 1122, 2161, 3970, 6181, 8796, 1370, 2961, 7172, 5483]


## 2. Autoencoder

Cuello de botella de 16 unidades: comprime la imagen a un vector y la reconstruye. Se entrena con
`EarlyStopping(restore_best_weights=True)`, de modo que se conserva el mejor punto y no el último.

In [3]:
autoencoder = build_autoencoder(tf)
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
inicio = time.perf_counter()
historia_ae = autoencoder.fit(x_train, x_train, validation_split=0.1, epochs=6, batch_size=256,
                              callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
                              verbose=0)
segundos_ae = time.perf_counter() - inicio
print("AE entrenado en %.2f s | pérdida final: %.4f | validación: %.4f"
      % (segundos_ae, historia_ae.history["loss"][-1], historia_ae.history["val_loss"][-1]))

reconstruccion_ae = autoencoder.predict(x_eval, verbose=0)
mse_ae = float(np.mean((x_eval - reconstruccion_ae) ** 2))
print("MSE de reconstrucción (1.000 ejemplos de prueba): %.5f" % mse_ae)

E0000 00:00:1793620573.700683   12909 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


AE entrenado en 9.08 s | pérdida final: 0.3060 | validación: 0.3071
MSE de reconstrucción (1.000 ejemplos de prueba): 0.02048


E0000 00:00:1793620582.599222   12909 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


## 3. VAE

La pérdida total es la reconstrucción más la divergencia KL. El espacio latente de 2 dimensiones
permite visualizar, pero **sus ejes no tienen significado interpretable**: no son "tipo de prenda"
ni "color".

In [4]:
class VAE(tf.keras.Model):
    # Misma definición que el script base del profesor (scripts/train_ae_vae.py).
    # El train_step calcula la pérdida total (reconstrucción binaria + divergencia KL) con un
    # GradientTape; sin este paso Keras no tendría ninguna pérdida que minimizar.

    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.reconstruction_tracker = tf.keras.metrics.Mean(name="reconstruction")
        self.kl_tracker = tf.keras.metrics.Mean(name="kl")

    @property
    def metrics(self):
        return [self.loss_tracker, self.reconstruction_tracker, self.kl_tracker]

    def train_step(self, data):
        images = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            mean, log_variance = self.encoder(images, training=True)
            epsilon = tf.random.normal(tf.shape(mean))
            latent = mean + tf.exp(0.5 * log_variance) * epsilon
            reconstruction = self.decoder(latent, training=True)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(tf.keras.losses.binary_crossentropy(images, reconstruction), axis=(1, 2)))
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(1 + log_variance - tf.square(mean) - tf.exp(log_variance), axis=1))
            total_loss = reconstruction_loss + kl_loss
        gradients = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))
        self.loss_tracker.update_state(total_loss)
        self.reconstruction_tracker.update_state(reconstruction_loss)
        self.kl_tracker.update_state(kl_loss)
        return {metric.name: metric.result() for metric in self.metrics}


encoder, decoder = build_encoder_decoder(tf, latent_dim=2)
vae = VAE(encoder, decoder)
vae.compile(optimizer="adam")
inicio = time.perf_counter()
historia_vae = vae.fit(x_train, epochs=8, batch_size=256, verbose=0)
segundos_vae = time.perf_counter() - inicio
print("VAE entrenado en %.2f s" % segundos_vae)

mean, log_variance = encoder.predict(x_eval, verbose=0)
reconstruccion_vae = decoder.predict(mean, verbose=0)
mse_vae = float(np.mean((x_eval - reconstruccion_vae) ** 2))
kl = float(-0.5 * np.mean(np.sum(1 + log_variance - np.square(mean) - np.exp(log_variance), axis=1)))
print("MSE de reconstrucción (VAE): %.5f | divergencia KL media: %.4f" % (mse_vae, kl))

VAE entrenado en 14.04 s


MSE de reconstrucción (VAE): 0.03375 | divergencia KL media: 6.3744


E0000 00:00:1793620596.786571   12909 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


## 4. Muestreo e interpolación

Una imagen generada **no** es automáticamente privada, original ni libre de sesgo: sin un análisis de
vecinos cercanos en el conjunto de entrenamiento no se puede descartar la memorización.

In [5]:
muestras = decoder.predict(np.random.default_rng(SEED).normal(size=(16, 2)), verbose=0)
rejilla = np.array([(1 - v) * np.array([-2.0, -1.0]) + v * np.array([2.0, 1.0]) for v in np.linspace(0, 1, 16)])
interpolacion = decoder.predict(rejilla, verbose=0)

originales = x_eval[panel_index]
figura, ejes = plt.subplots(4, 16, figsize=(16, 4.6))
for columna in range(16):
    for fila, coleccion in enumerate([originales, reconstruccion_ae[panel_index],
                                      reconstruccion_vae[panel_index], muestras]):
        ejes[fila, columna].imshow(coleccion[columna].squeeze(), cmap="gray")
        ejes[fila, columna].axis("off")
for fila, titulo in enumerate(["Originales (prueba)", "Reconstrucción AE", "Reconstrucción VAE", "Muestras del VAE"]):
    ejes[fila, 0].set_ylabel(titulo, fontsize=7)
figura.suptitle("Fashion-MNIST · 16 ejemplos de prueba elegidos con semilla 42 (panel no seleccionado)")
figura.tight_layout()
figura.savefig(REPORTS / "ejercicio06_panel.png", dpi=170, bbox_inches="tight")
plt.close(figura)

figura, ejes = plt.subplots(1, 16, figsize=(16, 1.5))
for eje, imagen in zip(ejes, interpolacion):
    eje.imshow(imagen.squeeze(), cmap="gray")
    eje.axis("off")
figura.suptitle("Interpolación en el espacio latente del VAE (16 puntos)")
figura.tight_layout()
figura.savefig(REPORTS / "ejercicio06_interpolacion.png", dpi=170, bbox_inches="tight")
plt.close(figura)
print("Figuras guardadas: ejercicio06_panel.png, ejercicio06_interpolacion.png")

Figuras guardadas: ejercicio06_panel.png, ejercicio06_interpolacion.png


## 5. Fidelidad, diversidad, memoria y costo

- **Fidelidad**: MSE de reconstrucción sobre 1.000 ejemplos de prueba.
- **Diversidad**: distancia media entre muestras del decoder y entre la muestra y su vecino más
  cercano en el conjunto de entrenamiento (indicador de posible memorización).
- **Memoria**: tamaño de los modelos serializados.
- **Costo**: tiempo de entrenamiento e inferencia.

In [6]:
# diversidad: distancia media entre muestras del decoder
latentes = np.random.default_rng(SEED).normal(size=(64, 2))
generadas = decoder.predict(latentes, verbose=0).reshape(64, -1)
distancias = [float(np.linalg.norm(generadas[i] - generadas[j])) for i in range(64) for j in range(i + 1, 64)]
diversidad_muestra = float(np.mean(distancias))

# posible memorización: distancia de cada muestra a su vecino más cercano en 1.000 ejemplos de prueba
plano_prueba = x_eval.reshape(len(x_eval), -1)
plano_generadas = generadas[:32]
vecinos = []
for fila in plano_generadas:
    distances = np.linalg.norm(plano_prueba - fila, axis=1)
    vecinos.append(float(distances.min()))
distancia_vecino = float(np.mean(vecinos))

autoencoder.save(MODELS / "autoencoder.keras")
decoder.save(MODELS / "decoder.keras")
import pickle

tamano_ae = len(pickle.dumps(autoencoder)) / 1024
tamano_decoder = len(pickle.dumps(decoder)) / 1024

inicio = time.perf_counter()
_ = decoder.predict(np.zeros((100, 2)), verbose=0)
ms_inferencia = (time.perf_counter() - inicio) * 1000 / 100

metricas = {
    "mse_reconstruccion_ae": mse_ae,
    "mse_reconstruccion_vae": mse_vae,
    "kl_media_vae": kl,
    "diversidad_media_entre_muestras": diversidad_muestra,
    "distancia_media_al_vecino_mas_cercano": distancia_vecino,
    "segundos_entrenamiento_ae": segundos_ae,
    "segundos_entrenamiento_vae": segundos_vae,
    "ms_inferencia_por_lote_100": ms_inferencia,
    "kb_modelo_ae": tamano_ae,
    "kb_decoder": tamano_decoder,
    "parametros_ae": int(autoencoder.count_params()),
    "parametros_decoder": int(decoder.count_params()),
    "seed": SEED,
}
(REPORTS / "ejercicio06_metricas.json").write_text(json.dumps(metricas, indent=2), encoding="utf-8")
for clave, valor in metricas.items():
    print(f"{clave:38s}: {valor if isinstance(valor, int) else round(valor, 5)}")

mse_reconstruccion_ae                 : 0.02048
mse_reconstruccion_vae                : 0.03375
kl_media_vae                          : 6.3744
diversidad_media_entre_muestras       : 4.8633
distancia_media_al_vecino_mas_cercano : 3.2987
segundos_entrenamiento_ae             : 9.08334
segundos_entrenamiento_vae            : 14.04421
ms_inferencia_por_lote_100            : 0.46382
kb_modelo_ae                          : 330.81445
kb_decoder                            : 415.80762
parametros_ae                         : 25888
parametros_decoder                    : 101520
seed                                  : 42


## 6. Curvas de entrenamiento

La lectura clave del manual: si la pérdida de entrenamiento baja y la de validación sube, hay
sobreajuste; si ambas quedan altas, subajuste o problema de datos.

In [7]:
figura, (eje1, eje2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
eje1.plot(historia_ae.history["loss"], label="entrenamiento")
eje1.plot(historia_ae.history["val_loss"], label="validación")
eje1.set_title("Autoencoder · pérdida binary crossentropy")
eje1.set_xlabel("época")
eje1.set_ylabel("pérdida")
eje1.legend(fontsize=8)
eje1.grid(alpha=0.3)

eje2.plot(historia_vae.history["loss"], label="total")
eje2.plot(historia_vae.history["reconstruction"], label="reconstrucción")
eje2.plot(historia_vae.history["kl"], label="KL")
eje2.set_title("VAE · pérdida total, reconstrucción y KL")
eje2.set_xlabel("época")
eje2.set_ylabel("pérdida")
eje2.legend(fontsize=8)
eje2.grid(alpha=0.3)
figura.tight_layout()
figura.savefig(REPORTS / "ejercicio06_curvas.png", dpi=170, bbox_inches="tight")
plt.close(figura)
print("Curvas guardadas en reports/ejercicio06_curvas.png")

Curvas guardadas en reports/ejercicio06_curvas.png


## 7. Interpretación y decisión

**Observación.** El AE reconstruye mejor (MSE 0,0205) que el VAE (0,0338): la penalización KL
compra un espacio muestreable a cambio de nitidez. La divergencia KL media (6,37) muestra que el
latente se aleja de la referencia normal, y con solo 2 dimensiones las proyecciones se agrupan y las
interpolaciones no siempre son plausibles.

**Evidencia.** Los paneles están construidos sobre una muestra con semilla fija (16 de 1.000 casos de
prueba), no sobre los mejores casos; la distancia media al vecino más cercano del conjunto de
prueba y la distancia media entre muestras cuantifican la diversidad sin poder descartar la
memorización.

**Decisión.** El VAE se acepta como generador ilustrativo del ejercicio y **no** como fuente de
imágenes utilizables: sin análisis de vecinos cercanos sobre el entrenamiento completo no se puede
afirmar privacidad, y el benchmark 28×28 en grises no es extrapolable a otros dominios. El AE se
prefiere cuando el objetivo es reconstruir. Ambos modelos quedan descritos en la Model Card del
repositorio, con sus límites y su costo.